# Learning Rate V.S. Temporal Difference Learning Steps Analysis

This notebook analyzes the relationship between learning rate and temporal difference learning steps.

## Objective

- Understand how different learning rates affect the number of training steps required for agent convergence in the ARC environment.
- Identify optimal learning rate in combination with TD learning steps for stable training.

## Methodology

### Dataset

Using the standard ARC dataset V1, we randomly select 100 training examples.

### Hyperparameter Grid

We will keep the default hyperparameters constant while varying only:

- Learning rate (e.g., 1e-3, 3e-4, 1e-4, 3e-5)
- Temporal difference learning steps (e.g., 1, 3, 5, 10)
- Training episodes = 10000
- Epochs = 10
- Batch size = 32

### Metrics to Track

- Training loss curves
- Reward progression
- Convergence speed
- Training stability (variance in loss/reward)
- Success rate (percentage of episodes where agent solves the task)


# Setup tarball with arc samples

In [1]:
import os

from boto3.session import Session
from sagemaker.core.training.configs import OutputDataConfig

from aws_sagemaker import utils
from aws_sagemaker.launch import SagemakerExperiment


sagemaker.config INFO - Not applying SDK defaults from location: /Library/Application Support/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /Users/andrestello/Library/Application Support/sagemaker/config.yaml


In [2]:
session = Session()

aws_sagemaker_bucket = os.getenv("AWS_SAGEMAKER_BUCKET")
sagemaker_role_arn = os.getenv("AWS_SAGEMAKER_ROLE_ARN")
experiment_location = "experiments/lr_td_analysis"

tar_s3_uri = utils.create_and_upload_arc_tar(
    session,
    "../experiments/lr_td_analysis/arc_sample.txt",
    aws_sagemaker_bucket,
    os.path.join(experiment_location, "arc_sample.tar.gz"),
    overwrite=False,
)
print("Dataset uploaded to: ", tar_s3_uri)

[12/26/25 16:03:28] WARNING  Object rlarcworld/experiments/lr_td_analysis/arc_sample.tar.gz already     ]8;id=24568;file:///Users/andrestello/Personal/RlArcProject/aws_sagemaker/utils.py\utils.py]8;;\:]8;id=134728;file:///Users/andrestello/Personal/RlArcProject/aws_sagemaker/utils.py#94\94]8;;\
                             exists; skipping upload.                                                              

Dataset uploaded to:  s3://rlarcworld/experiments/lr_td_analysis/arc_sample.tar.gz


In [4]:
experiment_name = "lr_1e3_nstep_1"
output_data_config = OutputDataConfig(
    s3_output_path=f"s3://{aws_sagemaker_bucket}/experiments/lr_td_analysis/{experiment_name}",
    compression_type="GZIP",
)
sagemaker_experiment = SagemakerExperiment(
    role_arn=sagemaker_role_arn,
    experiment=experiment_name,
    experiments_config="experiments/lr_td_analysis/experiments.yaml",
    output_data_config=output_data_config,
    epochs=1,  # short run
    instance_type="ml.g5.xlarge",  # single-GPU; adjust if you only have CPU quota
    instance_count=1,
    volume_size=50,  # GB
)
sagemaker_experiment.create_model_trainer()


[12/26/25 16:09:17] INFO     SageMaker Session:                                                        ]8;id=635572;file:///Users/andrestello/Personal/RlArcProject/aws_sagemaker/launch.py\launch.py]8;;\:]8;id=774040;file:///Users/andrestello/Personal/RlArcProject/aws_sagemaker/launch.py#85\85]8;;\

                    INFO       - role arn: arn:aws:iam::921868060019:role/SagemakerFullAccessExp       ]8;id=238258;file:///Users/andrestello/Personal/RlArcProject/aws_sagemaker/launch.py\launch.py]8;;\:]8;id=257129;file:///Users/andrestello/Personal/RlArcProject/aws_sagemaker/launch.py#86\86]8;;\

[12/26/25 16:09:18] INFO     Created S3 bucket: sagemaker-us-east-1-921868060019              ]8;id=353720;file:///Users/andrestello/Personal/RlArcProject/.venv/lib/python3.12/site-packages/sagemaker/core/helper/session_helper.py\session_helper.py]8;;\:]8;id=936031;file:///Users/andrestello/Personal/RlArcProject/.venv/lib/python3.12/site-packages/sagemaker/core/helper/session_helper.py#752\752]8;;\

                    INFO       - bucket: sagemaker-us-east-1-921868060019                              ]8;id=530747;file:///Users/andrestello/Personal/RlArcProject/aws_sagemaker/launch.py\launch.py]8;;\:]8;id=523067;file:///Users/andrestello/Personal/RlArcProject/aws_sagemaker/launch.py#87\87]8;;\

                    INFO       - region: us-east-1                                                     ]8;id=906453;file:///Users/andrestello/Personal/RlArcProject/aws_sagemaker/launch.py\launch.py]8;;\:]8;id=740632;file:///Users/andrestello/Personal/RlArcProject/aws_sagemaker/launch.py#88\88]8;;\

...

                    INFO     Compute: instance_type='ml.g5.xlarge' instance_count=1                   ]8;id=967030;file:///Users/andrestello/Personal/RlArcProject/aws_sagemaker/launch.py\launch.py]8;;\:]8;id=906875;file:///Users/andrestello/Personal/RlArcProject/aws_sagemaker/launch.py#101\101]8;;\
                             volume_size_in_gb=50 volume_kms_key_id=None                                           
                             keep_alive_period_in_seconds=None capacity_reservation_ids=None                       
                             instance_groups=None capacity_schedules_config=None                                   
                             training_plan_arn=None instance_placement_config=None                                 
                             enable_managed_spot_training=None                                                     

                    INFO     Training image URI:                                               ]8;id=929321;file:///Users/andrestello/Personal/RlArcProject/.venv/lib/python3.12/site-packages/sagemaker/train/model_trainer.py\model_trainer.py]8;;\:]8;id=918751;file:///Users/andrestello/Personal/RlArcProject/.venv/lib/python3.12/site-packages/sagemaker/train/model_trainer.py#537\537]8;;\
                             763104351884.dkr.ecr.us-east-1.amazonaws.com/pytorch-training:2.1                     
                             .0-gpu-py310                                                                          

                    INFO     Model trainer created: training_mode=<Mode.SAGEMAKER_TRAINING_JOB:       ]8;id=584484;file:///Users/andrestello/Personal/RlArcProject/aws_sagemaker/launch.py\launch.py]8;;\:]8;id=209350;file:///Users/andrestello/Personal/RlArcProject/aws_sagemaker/launch.py#179\179]8;;\
                             'SAGEMAKER_TRAINING_JOB'>                                                             
                             sagemaker_session=<sagemaker.core.helper.session_helper.Session object                
                             at 0x1085698b0>                                                                       
                             role='arn:aws:iam::921868060019:role/SagemakerFullAccessExp'                          
                             base_job_name='lr_1e3_nstep_1'                                                        
                             source_code=SourceCode(source_dir='/Users/andrestello/Personal/RlArcProj              
                             ect', requirements=None, entry_script='aws_sagemaker/train.py',                       
                             command=None, ignore_patterns=['*.ipynb', '.git', '.github', '.venv',                 
                             '**/__pycache__', 'tests'])                                                           
                             distributed=Torchrun(process_count_per_node=None, smp=None)                           
                             compute=Compute(instance_type='ml.g5.xlarge', instance_count=1,                       
                             volume_size_in_gb=50, volume_kms_key_id=None,                                         
                             keep_alive_period_in_seconds=None, capacity_reservation_ids=None,                     
                             instance_groups=None, capacity_schedules_config=None,                                 
                             training_plan_arn=None, instance_placement_config=None,                               
                             enable_managed_spot_training=None) networking=None                                    
                             stopping_condition=StoppingCondition(max_runtime_in_seconds=3600,                     
                             max_wait_time_in_seconds=<sagemaker.core.utils.utils.Unassigned object                
                             at 0x110801970>,                                                                      
                             max_pending_time_in_seconds=<sagemaker.core.utils.utils.Unassigned                    
                             object at 0x110801970>)                                                               
                             training_image='763104351884.dkr.ecr.us-east-1.amazonaws.com/pytorch-tra              
                             ining:2.1.0-gpu-py310' training_image_config=None algorithm_name=None                 
                             output_data_config=OutputDataConfig(s3_output_path='s3://rlarcworld/expe              
                             riments/lr_td_analysis/lr_1e3_nstep_1', kms_key_id=None,                              
                             compression_type='GZIP',                                                              
                             remove_job_name_from_s3_output_path=<sagemaker.core.utils.utils.Unassign              
                             ed object at 0x110801970>,                                                            
                             disable_model_upload=<sagemaker.core.utils.utils.Unassigned object at                 
                             0x110801970>, channels=<sagemaker.core.utils.utils.Unassigned object at               
                             0x110801970>) input_data_config=None checkpoint_config=None                           
                             training_input_mode='File' environment={'NCCL_ASYNC_ERROR_HANDLING'

                    INFO     Training image URI:                                               ]8;id=502624;file:///Users/andrestello/Personal/RlArcProject/.venv/lib/python3.12/site-packages/sagemaker/train/model_trainer.py\model_trainer.py]8;;\:]8;id=96583;file:///Users/andrestello/Personal/RlArcProject/.venv/lib/python3.12/site-packages/sagemaker/train/model_trainer.py#537\537]8;;\
                             763104351884.dkr.ecr.us-east-1.amazonaws.com/pytorch-training:2.1                     
                             .0-gpu-py310                                                                          

                    INFO     Model trainer created: training_mode=<Mode.SAGEMAKER_TRAINING_JOB:       ]8;id=929487;file:///Users/andrestello/Personal/RlArcProject/aws_sagemaker/launch.py\launch.py]8;;\:]8;id=573020;file:///Users/andrestello/Personal/RlArcProject/aws_sagemaker/launch.py#179\179]8;;\
                             'SAGEMAKER_TRAINING_JOB'>                                                             
                             sagemaker_session=<sagemaker.core.helper.session_helper.Session object                
                             at 0x1085698b0>                                                                       
                             role='arn:aws:iam::921868060019:role/SagemakerFullAccessExp'                          
                             base_job_name='lr_1e3_nstep_1'                                                        
                             source_code=SourceCode(source_dir='/Users/andrestello/Personal/RlArcProj              
                             ect', requirements=None, entry_script='aws_sagemaker/train.py',                       
                             command=None, ignore_patterns=['*.ipynb', '.git', '.github', '.venv',                 
                             '**/__pycache__', 'tests'])                                                           
                             distributed=Torchrun(process_count_per_node=None, smp=None)                           
                             compute=Compute(instance_type='ml.g5.xlarge', instance_count=1,                       
                             volume_size_in_gb=50, volume_kms_key_id=None,                                         
                             keep_alive_period_in_seconds=None, capacity_reservation_ids=None,                     
                             instance_groups=None, capacity_schedules_config=None,                                 
                             training_plan_arn=None, instance_placement_config=None,                               
                             enable_managed_spot_training=None) networking=None                                    
                             stopping_condition=StoppingCondition(max_runtime_in_seconds=3600,                     
                             max_wait_time_in_seconds=<sagemaker.core.utils.utils.Unassigned object                
                             at 0x110801970>,                                                                      
                             max_pending_time_in_seconds=<sagemaker.core.utils.utils.Unassigned                    
                             object at 0x110801970>)                                                               
                             training_image='763104351884.dkr.ecr.us-east-1.amazonaws.com/pytorch-tra              
                             ining:2.1.0-gpu-py310' training_image_config=None algorithm_name=None                 
                             output_data_config=OutputDataConfig(s3_output_path='s3://rlarcworld/expe              
                             riments/lr_td_analysis/lr_1e3_nstep_1', kms_key_id=None,                              
                             compression_type='GZIP',                                                              
                             remove_job_name_from_s3_output_path=<sagemaker.core.utils.utils.Unassign              
                             ed object at 0x110801970>,                                                            
                             disable_model_upload=<sagemaker.core.utils.utils.Unassigned object at                 
                             0x110801970>, channels=<sagemaker.core.utils.utils.Unassigned object at               
                             0x110801970>) input_data_config=None checkpoint_config=None                           
                             training_input_mode='File' environment={'NCCL_ASYNC_ERROR_HANDLING'

In [ ]:
sagemaker_experiment.train(tar_s3_uri)

[12/26/25 16:13:44] INFO     SageMaker Python SDK will collect telemetry to help us better  ]8;id=842806;file:///Users/andrestello/Personal/RlArcProject/.venv/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=958123;file:///Users/andrestello/Personal/RlArcProject/.venv/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py#91\91]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#confi                        
                             guring-and-using-defaults-with-the-sagemaker-python-sdk.                              

sagemaker.config INFO - Not applying SDK defaults from location: /Library/Application Support/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /Users/andrestello/Library/Application Support/sagemaker/config.yaml


[12/26/25 16:14:05] INFO     Creating training_job resource.                                     ]8;id=71525;file:///Users/andrestello/Personal/RlArcProject/.venv/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=50698;file:///Users/andrestello/Personal/RlArcProject/.venv/lib/python3.12/site-packages/sagemaker/core/resources.py#35539\35539]8;;\

                    WARNING  No region provided. Using default region.                                 ]8;id=726898;file:///Users/andrestello/Personal/RlArcProject/.venv/lib/python3.12/site-packages/sagemaker/core/utils/utils.py\utils.py]8;;\:]8;id=128849;file:///Users/andrestello/Personal/RlArcProject/.venv/lib/python3.12/site-packages/sagemaker/core/utils/utils.py#340\340]8;;\

                    INFO     Runs on sagemaker prod, region:us-east-1                                  ]8;id=938552;file:///Users/andrestello/Personal/RlArcProject/.venv/lib/python3.12/site-packages/sagemaker/core/utils/utils.py\utils.py]8;;\:]8;id=19426;file:///Users/andrestello/Personal/RlArcProject/.venv/lib/python3.12/site-packages/sagemaker/core/utils/utils.py#354\354]8;;\

/Users/andrestello/Personal/RlArcProject/.venv/lib/python3.12/site-packages/rich/live.py:231: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Output()